# Multi-R Examples — C Cases Matched to Multiple R Cases

A **simpler** companion to `investigate_c_to_multiple_r.ipynb`, built for one purpose:
surface **C Case <-> R Case numbers** you can look up directly in the NLRB source data files.

For each example it prints a **C Case number** and the **several R Case numbers** it was
matched to, with each case's **filing system** (CHIPS / CATS / NxGen), city, and dates for context.

Examples are sampled across:
- **filing systems** — CHIPS, CATS, NxGen (from the authoritative `data_source` column), and
- **group sizes** — C cases matched to 2, 3, 4-5, and 6+ R cases.

Region is already enforced in the matching, so every R case shares the C case's Region.

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

# This notebook lives in multi_r_per_c_investigation/; inputs live in the project root.
# The output (multi_r_examples_*.csv) is written alongside this notebook.
DATA_DIR = Path("..")
OUT_DIR = Path(".")

# Which matching output to inspect. Switch METHOD to "cluster" to use the cluster matches.
MATCH_FILES = {
    "fuzzy":   DATA_DIR / "rc_ac_matches.parquet",
    "cluster": DATA_DIR / "rc_ac_cluster_matches_20260517.parquet",
}
METHOD = "fuzzy"

# Authoritative filing-system label (CHIPS / CATS / NxGen) per case number.
R_FINAL = DATA_DIR / "merged_R_CASES_final.parquet"
C_FINAL = DATA_DIR / "merged_C_CASES_final.parquet"

SEED = 0          # reproducible sampling
N_PER_CELL = 2    # examples to print per (filing system x size) combination

## 1. Load matches and attach filing system (era)

The match output stores only case numbers and dates, so we join the `data_source`
column (CHIPS / CATS / NxGen) from the merged "final" tables by case number.

In [2]:
matches = pd.read_parquet(MATCH_FILES[METHOD])
print(f"{METHOD}: {len(matches):,} pairs | "
      f"{matches['c_case_number'].nunique():,} C cases | "
      f"{matches['r_case_number'].nunique():,} R cases")

r_src = (pd.read_parquet(R_FINAL, columns=["case_number", "data_source"])
           .drop_duplicates("case_number").set_index("case_number")["data_source"])
c_src = (pd.read_parquet(C_FINAL, columns=["case_number", "data_source"])
           .drop_duplicates("case_number").set_index("case_number")["data_source"])

matches["r_era"] = matches["r_case_number"].map(r_src)
matches["c_era"] = matches["c_case_number"].map(c_src)
print(f"era coverage -> C: {matches['c_era'].notna().mean()*100:.1f}%  "
      f"R: {matches['r_era'].notna().mean()*100:.1f}%")

fuzzy: 49,429 pairs | 45,824 C cases | 25,539 R cases
era coverage -> C: 100.0%  R: 100.0%


## 2. C Cases matched to multiple R Cases

One row per C case that links to >1 R case: its filing system, how many R cases,
and basic identifying fields.

In [3]:
n_rc = matches.groupby("c_case_number")["r_case_number"].nunique()
multi_ca = n_rc[n_rc > 1].index
multi = matches[matches["c_case_number"].isin(multi_ca)].copy()

groups = (multi.groupby("c_case_number")
                .agg(c_era=("c_era", "first"),
                     n_rc=("r_case_number", "nunique"),
                     c_company=("c_company_name", "first"),
                     c_city=("c_city", "first"),
                     c_date_filed=("c_date_filed", "first"))
                .reset_index())

print(f"C cases matched to >1 R case: {len(groups):,}\n")
print("By filing system (C case):")
print(groups["c_era"].value_counts().to_string())
print("\nBy number of R cases matched:")
print(groups["n_rc"].value_counts().sort_index().to_string())

C cases matched to >1 R case: 2,414

By filing system (C case):
c_era
CHIPS    1229
CATS      767
NxGen     418

By number of R cases matched:
n_rc
2     1856
3      351
4       82
5       29
6       55
7       11
8        6
9        4
14      15
15       2
16       1
17       2


## 3. Helper for printing a group

`print_group` shows the C case number and every R case number it matched, each tagged
with its filing system so you know which source file to open.

In [4]:
def size_bucket(n):
    if n == 2: return "2 R cases"
    if n == 3: return "3 R cases"
    if n <= 5: return "4-5 R cases"
    return "6+ R cases"

groups["size_bucket"] = groups["n_rc"].apply(size_bucket)

def _d(x):
    x = pd.to_datetime(x, errors="coerce")
    return x.date() if pd.notna(x) else "?"

def print_group(c_case):
    g = groups.loc[groups["c_case_number"] == c_case].iloc[0]
    print(f"C CASE  {c_case}   [{g['c_era']}]   ->  {g['n_rc']} R cases")
    print(f"        {g['c_company']}  |  {g['c_city']}  |  filed {_d(g['c_date_filed'])}")
    links = matches.loc[matches["c_case_number"] == c_case].sort_values("r_date_filed")
    for _, r in links.iterrows():
        print(f"          R  {r['r_case_number']}   [{str(r['r_era']):>5}]   "
              f"{str(r['r_city'])[:20]:20}   "
              f"filed {_d(r['r_date_filed'])}  closed {_d(r['r_date_closed'])}   "
              f"{r['match_method']}")
    print()

## 4. Examples by filing system x number of R cases

Walks every (CHIPS/CATS/NxGen) x (2 / 3 / 4-5 / 6+) combination and prints
`N_PER_CELL` example groups for each. Cells with no such C cases are noted as `(none)`.

In [5]:
ERAS = ["CHIPS", "CATS", "NxGen"]
BUCKETS = ["2 R cases", "3 R cases", "4-5 R cases", "6+ R cases"]

for era in ERAS:
    for bucket in BUCKETS:
        sub = groups[(groups["c_era"] == era) & (groups["size_bucket"] == bucket)]
        print("=" * 72)
        print(f"FILING SYSTEM = {era}   |   {bucket}   |   {len(sub):,} such C cases")
        print("=" * 72)
        if sub.empty:
            print("   (none)\n")
            continue
        pick = sub.sample(min(N_PER_CELL, len(sub)), random_state=SEED)
        for c_case in pick["c_case_number"]:
            print_group(c_case)

FILING SYSTEM = CHIPS   |   2 R cases   |   951 such C cases
C CASE  30-CA-014400   [CHIPS]   ->  2 R cases
        FELTNER PLUMBING AND HEATING  |  MARQUETTE  |  filed 1998-08-27
          R  30-RC-006036   [CHIPS]   MARQUETTE              filed 1998-08-21  closed 1998-10-23   exact
          R  30-RC-006037   [CHIPS]   MARQUETTE              filed 1998-08-21  closed 1998-10-23   exact

C CASE  20-CA-023685   [CHIPS]   ->  2 R cases
        QUEEN OF THE VALLEY HOSPITAL  |  NAPA  |  filed 1990-10-26
          R  20-RC-016612   [CHIPS]   NAPA                   filed 1990-04-05  closed 1995-03-14   exact
          R  20-RC-016656   [CHIPS]   NAPA                   filed 1990-08-08  closed 1991-07-05   exact

FILING SYSTEM = CHIPS   |   3 R cases   |   207 such C cases
C CASE  01-CA-030905   [CHIPS]   ->  3 R cases
        RHODE ISLAND HOSPITAL  |  PROVIDENCE  |  filed 1993-09-15
          R  01-RC-019972   [CHIPS]   PROVIDENCE             filed 1993-04-26  closed 1993-12-27   exact
     

## 5. Flat lookup table for the source-data cross-check

One row per multi-R C case, with all its R case numbers joined into a single string,
plus filing system and size. Written to CSV so you can filter/sort it while cross-checking
against the NLRB source files.

In [ ]:
rlist = (multi.sort_values("r_date_filed")
              .groupby("c_case_number")["r_case_number"]
              .apply(lambda s: ", ".join(pd.unique(s))))

lookup = groups.copy()
lookup["r_case_numbers"] = lookup["c_case_number"].map(rlist)
lookup = lookup[["c_case_number", "c_era", "n_rc", "size_bucket",
                 "c_company", "c_city", "r_case_numbers"]]

out = OUT_DIR / f"multi_r_examples_{METHOD}.csv"
lookup.to_csv(out, index=False)
print(f"wrote {out.name}  ({len(lookup):,} rows)")
lookup.head(20)